In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np

# 流场预测网络（输入：时间t + 插值样本x_t + 条件c；输出：流场v_t）
class FlowMatchingModel(nn.Module):
    def __init__(self, input_dim=4, cond_dim=8, hidden_dim=128, time_embed_dim=32):
        super().__init__()
        # 时间步嵌入（将标量t转为高维向量）
        self.time_emb = nn.Sequential(
            nn.Linear(1, time_embed_dim),
            nn.ReLU(),
            nn.Linear(time_embed_dim, time_embed_dim)
        )
        # 条件信息编码
        self.cond_emb = nn.Sequential(
            nn.Linear(cond_dim, time_embed_dim),
            nn.ReLU(),
            nn.Linear(time_embed_dim, time_embed_dim)
        )
        # 核心流场预测网络
        self.backbone = nn.Sequential(
            nn.Linear(input_dim + time_embed_dim + time_embed_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, input_dim)  # 输出维度=轨迹特征维度（4）
        )

    def forward(self, t, x_t, cond):
        """
        预测流场v_t(x_t, c)
        Args:
            t: [B,] 时间步（0~1）
            x_t: [B, T, 4] 插值样本
            cond: [B, cond_dim] 条件信息（如初始位置、场景特征）
        Returns:
            v_pred: [B, T, 4] 预测流场
        """
        # 时间步嵌入：[B,] → [B,1] → [B, time_embed_dim]
        t_emb = self.time_emb(t.unsqueeze(1))  # [B, time_embed_dim]
        # 条件嵌入：[B, cond_dim] → [B, time_embed_dim]
        c_emb = self.cond_emb(cond)            # [B, time_embed_dim]
        
        # 扩展嵌入维度到[B, T, D]，匹配x_t形状
        B, T, D = x_t.shape
        t_emb = t_emb.unsqueeze(1).repeat(1, T, 1)  # [B, T, time_embed_dim]
        c_emb = c_emb.unsqueeze(1).repeat(1, T, 1)  # [B, T, time_embed_dim]
        
        # 拼接输入：x_t + 时间嵌入 + 条件嵌入
        x_input = torch.cat([x_t, t_emb, c_emb], dim=-1)  # [B, T, 4 + 2*time_embed_dim]
        v_pred = self.backbone(x_input)                  # [B, T, 4]
        return v_pred


In [2]:
def flow_matching_loss(model, x1, cond, sigma=1e-4):
    """
    计算Flow Matching Loss（Exact Flow Matching）
    Args:
        model: FlowMatchingModel 实例
        x1: [B, T, 4] 目标样本（真实轨迹，x_1 ~ p_data）
        cond: [B, cond_dim] 条件信息
        sigma: 小噪声，提升数值稳定性
    Returns:
        loss: 标量，Flow Matching Loss
    """
    B, T, D = x1.shape
    device = x1.device
    
    # 1. 采样x0（简单分布：标准正态分布）
    x0 = torch.randn_like(x1, device=device)  # [B, T, 4], x0 ~ N(0,I)
    
    # 2. 采样时间步t（均匀分布U(0,1)）
    t = torch.rand(B, device=device)  # [B,]
    
    # 3. 计算插值样本x_t = (1-t)x0 + t x1（扩展t维度到[B,1,1]）
    t_expand = t.view(B, 1, 1)        # [B,1,1]
    x_t = (1 - t_expand) * x0 + t_expand * x1  # [B, T, 4]
    
    # 4. 添加小噪声（可选，提升稳定性）
    x_t = x_t + sigma * torch.randn_like(x_t)
    
    # 5. 计算真实流场v_t* = x1 - x0
    v_true = x1 - x0  # [B, T, 4]
    
    # 6. 模型预测流场v_pred
    v_pred = model(t, x_t, cond)
    
    # 7. 计算Loss（可选添加时间权重t）
    loss = (t.view(B,1,1) * (v_pred - v_true).square()).mean()  # 带时间权重的MSE
    # 基础版Loss（无权重）：loss = (v_pred - v_true).square().mean()
    return loss


In [3]:
# 模拟自动驾驶轨迹数据集
class TrajectoryDataset(Dataset):
    def __init__(self, num_samples=1000, seq_len=10, cond_dim=8):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.cond_dim = cond_dim
        # 生成模拟轨迹（x/y范围：0~10，速度：0~5，航向：0~π）
        self.trajectories = torch.rand(num_samples, seq_len, 4) * torch.tensor([10,10,5,np.pi])
        # 生成模拟条件（如初始位置、场景类型）
        self.conditions = torch.rand(num_samples, cond_dim)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.trajectories[idx], self.conditions[idx]

# 训练配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
epochs = 50
lr = 1e-4

# 初始化数据集、模型、优化器
dataset = TrajectoryDataset(num_samples=1000)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
model = FlowMatchingModel(input_dim=4, cond_dim=8).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

# 训练循环
model.train()
for epoch in range(epochs):
    total_loss = 0.0
    for x1, cond in dataloader:
        x1 = x1.to(device)
        cond = cond.to(device)
        
        # 计算Flow Matching Loss
        loss = flow_matching_loss(model, x1, cond)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_loss:.6f}")

print("训练完成！")


Epoch [5/50], Avg Loss: 2.127573
Epoch [10/50], Avg Loss: 1.613807
Epoch [15/50], Avg Loss: 1.366125
Epoch [20/50], Avg Loss: 1.309958
Epoch [25/50], Avg Loss: 1.232584
Epoch [30/50], Avg Loss: 1.188793
Epoch [35/50], Avg Loss: 1.174814
Epoch [40/50], Avg Loss: 1.145059
Epoch [45/50], Avg Loss: 1.134714
Epoch [50/50], Avg Loss: 1.154374
训练完成！


In [4]:
def flow_matching_sample(model, cond, seq_len=10, num_steps=100, device="cuda"):
    """
    从流场生成样本（ODE积分）
    Args:
        model: 训练好的FlowMatchingModel
        cond: [B, cond_dim] 条件信息
        seq_len: 轨迹长度
        num_steps: ODE积分步数
    Returns:
        x1_pred: [B, seq_len, 4] 生成的轨迹
    """
    B = cond.shape[0]
    # 初始化x0 ~ N(0,I)
    x_t = torch.randn(B, seq_len, 4, device=device)
    dt = 1.0 / num_steps  # 积分步长
    
    model.eval()
    with torch.no_grad():
        for step in range(num_steps):
            t = torch.ones(B, device=device) * (step * dt)  # 当前时间步
            v_pred = model(t, x_t, cond)                    # 预测流场
            x_t = x_t + v_pred * dt                         # 欧拉积分更新
    
    return x_t  # x_t ≈ x1（目标分布样本）

# 示例：生成10条轨迹
cond = torch.rand(10, 8).to(device)
generated_trajectories = flow_matching_sample(model, cond, seq_len=10)
print("生成轨迹形状：", generated_trajectories.shape)  # [10, 10, 4]


生成轨迹形状： torch.Size([10, 10, 4])
